## Предобработка данных

### Разметка сплитов

In [1]:
import pandas as pd 
import numpy as np
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass

DATA_ROOT = Path("datasets/")

In [2]:
@dataclass 
class Config:
    seed: int = 42 
    n_folds: int = 5 
    val_splits: list[str] = None 

def load_meta():
    train_log = pd.read_csv(DATA_ROOT / "train_log.csv")
    test_log = pd.read_csv(DATA_ROOT / "test_log.csv")
    return train_log, test_log

def get_splits(train_log: pd.DataFrame, nfolds: int = 5):
    splits, folds = sorted(train_log["split"].unique()), []
    for i in range(nfolds): folds.append(splits[i::nfolds])
    return folds 

def get_fold_indices(train_log: pd.DataFrame, val_splits: list[str]):
    is_val = train_log["split"].isin(val_splits)
    train_idx = train_log.index[~is_val].to_numpy()
    val_idx = train_log.index[is_val].to_numpy()
    return train_idx, val_idx

In [3]:
train_log, test_log = load_meta()
folds = get_splits(train_log, nfolds=5)
print('folds:', folds)

folds: [['split_01', 'split_06', 'split_11', 'split_16'], ['split_02', 'split_07', 'split_12', 'split_17'], ['split_03', 'split_08', 'split_13', 'split_18'], ['split_04', 'split_09', 'split_14', 'split_19'], ['split_05', 'split_10', 'split_15', 'split_20']]


### Загрузка кривых по нужным splits

In [6]:
class Curver:
    def load_lightcurves_for_splits(self, splits: list[str], kind: str) -> pd.DataFrame:
        parts = []
        for split in splits:
            fname = 'train_full_lightcurves.csv' if kind == "train" else "test_full_lightcurves.csv"
            path = DATA_ROOT / split / fname
            df = pd.read_csv(path)
            
            df["splits"] = split 
            parts.append(df)

        lc = pd.concat(parts, ignore_index=True)
        return lc 

    def clean_lightcurves(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc = lc[lc["Flux"].notna()]
        lc["Time (MJD)"] = lc["Time (MJD)"].astype(float)
        lc["Flux"] = lc["Flux"].astype(float)
        lc["Flux_err"] = lc["Flux_err"].astype(float)
        return lc 

    def add_time_features(self, lc: pd.DataFrame) -> pd.DataFrame:
        lc = lc.copy()
        lc["t0"] = lc.groupby("object_id")["Time (MJD)"].transform("min")
        lc["dt"] = lc["Time (MJD)"] - lc["t0"]
        return lc 

In [ ]:
if __name__ == "__main__":
    curve = Curver()

    val_splits = folds[0]
    train_idx, val_idx = get_fold_indices(train_log, val_splits)

    # мета-таблицы
    train_meta = train_log.iloc[train_idx].copy()
    val_meta = train_log.iloc[val_idx].copy()

    # lightcurves
    train_lc = curve.load_lightcurves_for_splits(val_splits, kind="train")
    train_splits = [s for s in train_log["split"].unique() if s not in val_splits]
    train_lc = curve.load_lightcurves_for_splits(train_splits, kind="train")
    val_lc = curve.load_lightcurves_for_splits(val_splits, kind="train")

    train_lc = curve.add_time_features(curve.clean_lightcurves(train_lc))
    val_lc = curve.add_time_features(curve.clean_lightcurves(val_lc))
